In [1]:
import functools
from dust3r.demo import get_reconstructed_scene
from torch.nn.functional import mse_loss
import numpy as np

/home/kojogyaase/anaconda3/envs/mast3r-slam/lib/python3.11/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/home/kojogyaase/anaconda3/envs/mast3r-slam/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/kojogyaase/Projects/Research/3D_Recon/dependencies/dust3r/dust3r/cloud_opt/base_opt.py:275: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(enabled=False)


In [2]:
import os
import torch
import tempfile

from dust3r.model import AsymmetricCroCo3DStereo
from dust3r.demo import get_args_parser, main_demo, set_print_with_timestamp
from dust3r.inference import inference
from dust3r.model import AsymmetricCroCo3DStereo
from dust3r.utils.image import load_images
from dust3r.image_pairs import make_pairs
import matplotlib.pyplot as pl

from dust3r.inference import inference
from dust3r.image_pairs import make_pairs
from dust3r.utils.image import load_images, rgb
from dust3r.utils.device import to_numpy
from dust3r.viz import add_scene_cam, CAM_COLORS, OPENGL, pts3d_to_trimesh, cat_meshes
from dust3r.cloud_opt import global_aligner, GlobalAlignerMode

pl.ion()

torch.backends.cuda.matmul.allow_tf32 = True  # for gpu >= Ampere and pytorch >= 1.12
device="cuda"
batch_size=2
model_name="DUSt3R_ViTLarge_BaseDecoder_512_dpt"
weights="/home/kojogyaase/Projects/Research/3D_Recon/dependencies/dust3r/checkpoints/DUSt3R_ViTLarge_BaseDecoder_512_dpt.pth"
if weights is not None:
    weights_path = weights
else:
    weights_path = "naver/" + model_name
model = AsymmetricCroCo3DStereo.from_pretrained(weights_path).to(device)


... loading model from /home/kojogyaase/Projects/Research/3D_Recon/dependencies/dust3r/checkpoints/DUSt3R_ViTLarge_BaseDecoder_512_dpt.pth
instantiating : AsymmetricCroCo3DStereo(enc_depth=24, dec_depth=12, enc_embed_dim=1024, dec_embed_dim=768, enc_num_heads=16, dec_num_heads=12, pos_embed='RoPE100', patch_embed_cls='PatchEmbedDust3R', img_size=(512, 512), head_type='dpt', output_mode='pts3d', depth_mode=('exp', -inf, inf), conf_mode=('exp', 1, inf), landscape_only=False)
<All keys matched successfully>


In [3]:
import glob
# Load all images
# gt_depths= load_images(list(glob.glob("/home/kojogyaase/Projects/Research/3D_Recon/dependencies/dust3r/data/rgbd_dataset_freiburg1_xyz/depth/*.png"))[:5],grayscale=True, size=512)
# imgs = load_images(list(glob.glob("/home/kojogyaase/Projects/Research/3D_Recon/dependencies/dust3r/data/rgbd_dataset_freiburg1_xyz/rgb/*.png"))[:5], size=512)

base_path="/home/kojogyaase/Projects/Research/3D_Recon/dependencies/dust3r/data/rgbd_dataset_freiburg1_xyz/depth"

gt_depths=[
    base_path+"/1305031102.160407.png",
    base_path+"/1305031102.194330.png",
    base_path+"/1305031102.226738.png",
    base_path+"/1305031102.262886.png",
    base_path+"/1305031102.295279.png",
]
gt_depths= load_images(gt_depths,grayscale=True, size=512)
base_path="/home/kojogyaase/Projects/Research/3D_Recon/dependencies/dust3r/data/rgbd_dataset_freiburg1_xyz/rgb"

imgs=[
    base_path+"/1305031102.175304.png",
    base_path+"/1305031102.211214.png",
    base_path+"/1305031102.243211.png",
    base_path+"/1305031102.275326.png",
    base_path+"/1305031102.311267.png",
]
imgs = load_images(imgs, size=512)


>> Loading a list of 5 images
 - adding /home/kojogyaase/Projects/Research/3D_Recon/dependencies/dust3r/data/rgbd_dataset_freiburg1_xyz/depth/1305031102.160407.png with resolution 640x480 --> 512x384
 - adding /home/kojogyaase/Projects/Research/3D_Recon/dependencies/dust3r/data/rgbd_dataset_freiburg1_xyz/depth/1305031102.194330.png with resolution 640x480 --> 512x384
 - adding /home/kojogyaase/Projects/Research/3D_Recon/dependencies/dust3r/data/rgbd_dataset_freiburg1_xyz/depth/1305031102.226738.png with resolution 640x480 --> 512x384
 - adding /home/kojogyaase/Projects/Research/3D_Recon/dependencies/dust3r/data/rgbd_dataset_freiburg1_xyz/depth/1305031102.262886.png with resolution 640x480 --> 512x384
 - adding /home/kojogyaase/Projects/Research/3D_Recon/dependencies/dust3r/data/rgbd_dataset_freiburg1_xyz/depth/1305031102.295279.png with resolution 640x480 --> 512x384
 (Found 5 images)
>> Loading a list of 5 images
 - adding /home/kojogyaase/Projects/Research/3D_Recon/dependencies/dust3

In [4]:
# recon_fun = functools.partial(get_reconstructed_scene, tmpdirname, model, device, silent, image_size)
silent=False
schedule="cosine"
niter=300
# 

pairs = make_pairs(imgs, scene_graph='complete', prefilter=None, symmetrize=True)
output = inference(pairs, model, device, batch_size=batch_size)

mode = GlobalAlignerMode.PointCloudOptimizer if len(imgs) > 2 else GlobalAlignerMode.PairViewer
scene = global_aligner(output, device=device, mode=mode, verbose=not silent)
lr = 0.01

if mode == GlobalAlignerMode.PointCloudOptimizer:
    loss = scene.compute_global_alignment(init='mst', niter=niter, schedule=schedule, lr=lr)

# also return rgb, depth and confidence imgs
# depth is normalized with the max value for all images
# we apply the jet colormap on the confidence maps
rgbimg = scene.imgs
depths = to_numpy(scene.get_depthmaps())
confs = to_numpy([c for c in scene.im_conf])
cmap = pl.get_cmap('jet')
depths_max = max([d.max() for d in depths])
depths = [d / depths_max for d in depths]
confs_max = max([d.max() for d in confs])
confs = [cmap(d / confs_max) for d in confs]

imgs = []
depth_rgb=[]
for i in range(len(rgbimg)):
    imgs.append(rgbimg[i])
    imgs.append(rgb(depths[i]))
    depth_rgb.append(depths[i])
    imgs.append(rgb(confs[i]))


>> Inference with model on 20 image pairs


  0%|          | 0/10 [00:00<?, ?it/s]/home/kojogyaase/Projects/Research/3D_Recon/dependencies/dust3r/dust3r/inference.py:44: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=bool(use_amp)):
/home/kojogyaase/Projects/Research/3D_Recon/dependencies/dust3r/dust3r/model.py:205: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
/home/kojogyaase/Projects/Research/3D_Recon/dependencies/dust3r/dust3r/inference.py:48: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
100%|██████████| 10/10 [00:06<00:00,  1.45it/s]


 init edge (3*,2*) score=np.float64(55.06681442260742)
 init edge (0*,3) score=np.float64(54.2289924621582)
 init edge (4*,3) score=np.float64(49.96848678588867)
 init edge (0,1*) score=np.float64(45.33037185668945)
 init loss = 0.0067838262766599655
Global alignement - optimizing for:
['pw_poses', 'im_depthmaps', 'im_poses', 'im_focals']


100%|██████████| 300/300 [00:12<00:00, 23.25it/s, lr=1.27413e-06 loss=0.00356654]


In [6]:
gt_depths=torch.stack(list(map(lambda x:x["img"],gt_depths))).squeeze()

In [8]:
depth_rgb=torch.from_numpy(np.stack(depth_rgb))


In [ ]:
import torchvision

output=torch.concat([depth_rgb,gt_depths],dim=1).reshape((768*5,512))
torchvision.utils.save_image(output,"output.png")



In [ ]:
# https://arxiv.org/pdf/2312.14132
abs_rl=torch.abs(gt_depths - depth_rgb)/depth_rgb

In [ ]:
abs_rl

tensor(0.8950)